# Chapter 3 — Normalisation and Pigment Abundance Maps

This notebook robustly rescales the four indices to a common `[0, 1]`
range, calculates relative pigment-proxy contributions, creates a pigment
composite, supports interactive thresholding, and overlays the result on a
geographic basemap.

**Important:** percentile normalisation and relative contributions are
scene-specific. They do not represent absolute pigment concentrations.


In [ ]:
# # Run this cell once in a new environment, then restart the kernel if prompted.
# %pip install -r requirements.txt


In [ ]:
# cab = ds["CIRE"] # Indices used: CIRE as chlorophyll proxy
# anth = ds["ARI2"] # ARI2 (mARI) as anthocyanin proxy
# ccar = ds["CRI2"] # CRI2 as carotenoids proxy 

## 1. Load libraries and shared workshop functions


In [ ]:
from pathlib import Path
import sys
import rioxarray as rxr

import dask
import dask.array as dask_array
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
import ipywidgets as widgets

from IPython.display import display, Markdown, clear_output

pio.renderers.default = "plotly_mimetype+notebook_connected"

# Start a local Dask Client so workers and task activity are visible in the
# notebook. Threads avoid spawning extra Python processes and keep the worker
# setup reliable in Windows-based Jupyter environments.
from dask.distributed import Client, LocalCluster

DASK_WORKERS = 4
cluster = LocalCluster(
    n_workers=DASK_WORKERS,
    threads_per_worker=2,
    processes=False,
    dashboard_address=":0",
)
client = Client(cluster, set_as_default=True)
print(f"Dask scheduler: {client.scheduler.address}")
print(f"Dask workers: {len(client.scheduler_info()['workers'])}")
print(f"Dask dashboard: {client.dashboard_link}")
client

# The notebooks and workshop_utils.py should remain in the same folder.
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "workshop_utils.py").exists():
    raise FileNotFoundError(
        "Open Jupyter in the converted workshop folder so workshop_utils.py "
        "and requirements.txt are beside this notebook."
    )
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from plotly.subplots import make_subplots
from workshop_utils import (
    BAND_INFO, pstretch, array_to_png_b64, to_uint8, WORKSHOP_INDICES, INDEX_INFO, 
    interactive_raster_explorer, plot_raster_distribution,
    projected_plot_context, make_projected_image, apply_projected_axes, compute_indices, plot_indices, save_indices_to_tiff, 
)

# Set the path to the GeoTIFF stored on your computer.
# Expected workshop layout: notebook_folder/data/your_raster.tif
RASTER_FILENAME = "NTABRT0007-20240828-ms-cog-subset3.tif"
RASTER_PATH = NOTEBOOK_DIR / "data" / RASTER_FILENAME

# _HERE = Path(__file__).resolve().parent
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"

# Also allow the TIFF to sit beside the notebook.
if not RASTER_PATH.exists():
    local_file = NOTEBOOK_DIR / RASTER_FILENAME
    if local_file.exists():
        RASTER_PATH = local_file

if not RASTER_PATH.is_file():
    raise FileNotFoundError(
        f"Could not find the raster at {RASTER_PATH}. "
        "Edit RASTER_FILENAME / RASTER_PATH above to match the location "
        "of your local GeoTIFF."
    )

print(f"Local raster: {RASTER_PATH.resolve()}")
print(f"Dask scheduler: {client.scheduler.address}")
print(f"Dask workers: {len(client.scheduler_info()['workers'])}")


In [ ]:
# from pathlib import Path
# import sys

# import numpy as np
# import pandas as pd
# import plotly.graph_objects as go
# import plotly.express as px
# import plotly.io as pio
# import ipywidgets as widgets
# from IPython.display import display, Markdown, clear_output

# pio.renderers.default = "plotly_mimetype+notebook_connected"

# # The notebooks and workshop_utils.py should remain in the same folder.
# NOTEBOOK_DIR = Path.cwd()
# if not (NOTEBOOK_DIR / "workshop_utils.py").exists():
#     raise FileNotFoundError(
#         "Open Jupyter in the converted workshop folder so workshop_utils.py "
#         "and requirements.txt are beside this notebook."
#     )
# if str(NOTEBOOK_DIR) not in sys.path:
#     sys.path.insert(0, str(NOTEBOOK_DIR))

from plotly.subplots import make_subplots
from workshop_utils import (
    INDEX_INFO, WORKSHOP_INDICES, DATA_PATH, load_data, compute_indices,
    plot_indices, save_indices_to_tiff, pstretch, minmax, array_to_png_b64,
    projected_plot_context, make_projected_heatmap, make_projected_image,
    apply_projected_axes, interactive_raster_explorer,
)

# This chapter uses only these four indices.
ANALYSIS_INDICES = ["NDVI", "CIRE", "ARI2", "CRI2"]

print(f"Data file: {DATA_PATH}")


## 2. Load data and calculate indices

This chapter uses four indices calculated from the available MicaSense
RedEdge-P Dual bands:

| Index | Equation | Role |
|---|---|---|
| NDVI | $(R_{842}-R_{668})/(R_{842}+R_{668})$ | Vegetation mask/reference |
| CIRE | $(R_{842}/R_{705})-1$ | Chlorophyll proxy |
| ARI2 | $R_{842}[(1/R_{560})-(1/R_{705})]$ | Anthocyanin proxy |
| CRI2 | $(1/R_{531})-(1/R_{705})$ | Carotenoid proxy |

The ARI2 and CRI2 equations use the nearest available sensor bands to the
nominal wavelengths used by these indices.


In [ ]:
# Open the GeoTIFF as lazy Dask chunks. Only chunks needed by a later
# calculation are read into memory. Keep one band per chunk to bound working RAM.
RASTER_CHUNKS = {"band": 1, "x": 512, "y": 512}
raster = rxr.open_rasterio(
    RASTER_PATH,
    masked=True,
    chunks=RASTER_CHUNKS,
    cache=False,
    lock=True,
).astype("float32")

expected_band_names = [band["var"] for band in BAND_INFO]
if raster.sizes.get("band") != len(expected_band_names):
    raster.close()
    raise ValueError(
        f"Expected {len(expected_band_names)} bands, found "
        f"{raster.sizes.get('band')} in {RASTER_PATH.name}."
    )

# The supplied image uses zero as background/NoData. Keep values lazy while
# masking those pixels and converting the band dimension to named variables.
raster = raster.assign_coords(band=expected_band_names)
raster = raster.where(np.isfinite(raster) & (raster > 0))
ds = raster.to_dataset(dim="band")
ds.attrs.update({
    "filename": RASTER_PATH.name,
    "source_path": str(RASTER_PATH.resolve()),
    "instrument": "MicaSense RedEdge-P Dual",
    "image_width_px": int(raster.sizes["x"]),
    "image_height_px": int(raster.sizes["y"]),
    "pixel_size_m": float(abs(raster.rio.resolution()[0])),
    "crs": str(raster.rio.crs),
    "transform": tuple(raster.rio.transform()),
    "bounds": tuple(float(v) for v in raster.rio.bounds()),
    "dask_chunks": str(RASTER_CHUNKS),
})
for band in BAND_INFO:
    ds[band["var"]].attrs.update({
        "long_name": band["name"],
        "centre_wavelength_nm": band["nm"],
        "fwhm_nm": band["fwhm"],
    })

ds


In [ ]:
ds = compute_indices(ds,
    names=ANALYSIS_INDICES,
)
print("Data loaded and indices calculated:", ", ".join(ANALYSIS_INDICES))
ds

# ds = compute_indices(load_data(DATA_PATH))
# ds


## 2.5  NDVI vegetation mask

A single hard-coded threshold controls which pixels are included in **all**
downstream analysis.  Non-vegetation pixels are set to `NaN` in every index
immediately, so every later cell (normalisation, composites, pair-plots,
export) automatically operates on the masked scene only.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# NDVI THRESHOLD  — change this one value to adjust the vegetation mask
# ─────────────────────────────────────────────────────────────────────────────
NDVI_THRESHOLD = 0.30   # pixels with NDVI ≤ threshold become NaN everywhere

ndvi_raw = ds["NDVI"].values.astype("float32")
ndvi_mask = np.isfinite(ndvi_raw) & (ndvi_raw > NDVI_THRESHOLD)

total_px   = ndvi_raw.size
kept_px    = int(ndvi_mask.sum())
masked_px  = total_px - kept_px

print(f"NDVI threshold      : {NDVI_THRESHOLD}")
print(f"Total pixels        : {total_px:,}")
print(f"Vegetation pixels   : {kept_px:,}  ({100*kept_px/total_px:.1f} %)")
print(f"Masked-out pixels   : {masked_px:,}  ({100*masked_px/total_px:.1f} %)")

# Apply mask to every index in the dataset
for _name in ANALYSIS_INDICES:
    _arr = ds[_name].values.astype("float32")
    _arr[~ndvi_mask] = np.nan
    ds[_name].values[:] = _arr

print("\nNDVI mask applied to:", ", ".join(ANALYSIS_INDICES))
print("All subsequent cells use vegetation pixels only.")


In [ ]:
## Do indices plotly hostogram plot here

In [ ]:
# import plotly.graph_objects as go

# vals = ds["NDVI"].values.ravel()

# fig = go.Figure(
#     data=go.Histogram(x=vals, nbinsx=150, marker_color="#1f77b4")
# )

# fig.update_layout(
#     title="Histogram of NDVI",
#     xaxis_title="NDVI value",
#     yaxis_title="Frequency",
#     width=800,
#     height=500
# )

# fig.show()



import plotly.graph_objects as go
import numpy as np

vals = ds["NDVI"].values.ravel()

ndvi_min = float(np.nanmin(vals))
ndvi_max = float(np.nanmax(vals))
ndvi_median = float(np.nanmedian(vals))

fig = go.Figure(
    data=go.Histogram(x=vals, nbinsx=150, marker_color="#1f77b4", name="NDVI")
)

# Add vertical lines for min, max, median
fig.add_vline(
    x=ndvi_min,
    line_dash="dash",
    line_color="red",
    line_width=2,
    annotation_text=f"min: {ndvi_min:.3f}",
    annotation_position="top"
)

fig.add_vline(
    x=ndvi_max,
    line_dash="dash",
    line_color="green",
    line_width=2,
    annotation_text=f"max: {ndvi_max:.3f}",
    annotation_position="top"
)

fig.add_vline(
    x=ndvi_median,
    line_dash="solid",
    line_color="black",
    line_width=2,
    annotation_text=f"median: {ndvi_median:.3f}",
    annotation_position="top"
)

fig.update_layout(
    title=f"Histogram of NDVI — vegetation pixels only (threshold = {NDVI_THRESHOLD})",
    xaxis_title="NDVI value",
    yaxis_title="Frequency",
    width=900,
    height=550,
    showlegend=False
)

fig.show()

In [ ]:
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots

# # Assume your dataset is loaded as `ds`
# bands = ["NDVI", "CIRE", "ARI2", "CRI2"]
# colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

# fig = make_subplots(
#     rows=2,
#     cols=2,
#     subplot_titles=bands,
#     horizontal_spacing=0.12,
#     vertical_spacing=0.12
# )

# # Map band index to (row, col) in a 2x2 grid
# positions = [(1, 1), (1, 2), (2, 1), (2, 2)]

# for i, band in enumerate(bands):
#     r, c = positions[i]
#     vals = ds[band].values.ravel()
#     fig.add_trace(
#         go.Histogram(x=vals, nbinsx=150, marker_color=colors[i], name=band),
#         row=r,
#         col=c
#     )

# fig.update_layout(
#     height=700,
#     width=1200,
#     showlegend=False,
#     title_text="Histograms of NDVI, CIRE, ARI2, and CRI2",
#     title_x=0.5
# )

# fig.update_xaxes(title_text="Value", row=1, col=1)
# fig.update_xaxes(title_text="Value", row=1, col=2)
# fig.update_xaxes(title_text="Value", row=2, col=1)
# fig.update_xaxes(title_text="Value", row=2, col=2)

# fig.update_yaxes(title_text="Frequency", row=1, col=1)
# fig.update_yaxes(title_text="Frequency", row=2, col=1)

# fig.show()





import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

bands = ["NDVI", "CIRE", "ARI2", "CRI2"]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
positions = [(1, 1), (1, 2), (2, 1), (2, 2)]

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=bands,
    horizontal_spacing=0.12,
    vertical_spacing=0.12
)

for i, band in enumerate(bands):
    r, c = positions[i]
    vals = ds[band].values.ravel()

    b_min = float(np.nanmin(vals))
    b_max = float(np.nanmax(vals))
    b_median = float(np.nanmedian(vals))

    # Histogram
    fig.add_trace(
        go.Histogram(x=vals, nbinsx=150, marker_color=colors[i], name=band),
        row=r,
        col=c
    )

    # Min line
    fig.add_vline(
        x=b_min,
        line_dash="dash",
        line_color="red",
        line_width=1.5,
        annotation_text=f"min: {b_min:.3g}",
        annotation_position="top",
        row=r,
        col=c
    )

    # Max line
    fig.add_vline(
        x=b_max,
        line_dash="dash",
        line_color="green",
        line_width=1.5,
        annotation_text=f"max: {b_max:.3g}",
        annotation_position="top",
        row=r,
        col=c
    )

    # Median line
    fig.add_vline(
        x=b_median,
        line_dash="solid",
        line_color="black",
        line_width=1.5,
        annotation_text=f"median: {b_median:.3g}",
        annotation_position="top",
        row=r,
        col=c
    )

fig.update_layout(
    height=700,
    width=1200,
    showlegend=False,
    title_text="Histograms of NDVI, CIRE, ARI2, and CRI2 with Min, Max, Median",
    title_x=0.5
)

fig.update_xaxes(title_text="Value", row=1, col=1)
fig.update_xaxes(title_text="Value", row=1, col=2)
fig.update_xaxes(title_text="Value", row=2, col=1)
fig.update_xaxes(title_text="Value", row=2, col=2)

fig.update_yaxes(title_text="Frequency", row=1, col=1)
fig.update_yaxes(title_text="Frequency", row=2, col=1)

fig.show()

## 3. Robustly normalise each index to [0, 1]

Each index is scaled using its 2nd and 98th percentiles:

$$x' = \operatorname{clip}\left(\frac{x-P_2}{P_{98}-P_2},0,1\right).$$

The pigment proxies are then divided by their pixel-wise total to obtain
relative contributions.


## Calculate pixel-wise relative pigment-proxy contributions

Let the robust-normalised proxies be $A$ for `ARI2`, $C_h$ for `CIRE`, and $C_a$ for `CRI2`. The relative components are:

$$A_{rel}=\frac{A}{A+C_h+C_a},\qquad
C_{h,rel}=\frac{C_h}{A+C_h+C_a},\qquad
C_{a,rel}=\frac{C_a}{A+C_h+C_a}.$$

Pixels with missing inputs or a near-zero total are retained as NoData. A separate `NDVI > 0.20` mask limits maps and exports to vegetation.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Normalisation operates only on vegetation pixels (NDVI mask already applied).
# ─────────────────────────────────────────────────────────────────────────────
EPS = 1e-6


def robust_normalize(array, qlow=2, qhigh=98):
    """P2-P98 normalisation that retains invalid pixels as NaN."""
    array = np.asarray(array, dtype=np.float32)
    output = np.full(array.shape, np.nan, dtype=np.float32)

    valid = np.isfinite(array)
    if not valid.any():
        return output

    low, high = np.percentile(array[valid], [qlow, qhigh])
    if high - low <= EPS:
        output[valid] = 0.0
        return output

    output[valid] = np.clip(
        (array[valid] - low) / (high - low),
        0,
        1,
    )
    return output


# Robustly normalise only the four indices used in this chapter.
normed = {}
rows = []

for name in ANALYSIS_INDICES:
    arr = ds[name].values.astype("float32")
    nrm = robust_normalize(arr)
    normed[name] = nrm

    valid_raw = arr[np.isfinite(arr)]
    valid_nrm = nrm[np.isfinite(nrm)]

    rows.append({
        "Index": name,
        "Raw min": round(float(valid_raw.min()), 4),
        "P2": round(float(np.percentile(valid_raw, 2)), 4),
        "P98": round(float(np.percentile(valid_raw, 98)), 4),
        "Raw max": round(float(valid_raw.max()), 4),
        "Normalised min": round(float(valid_nrm.min()), 4),
        "Normalised max": round(float(valid_nrm.max()), 4),
        "Normalised mean": round(float(valid_nrm.mean()), 4),
    })

normalisation_stats = pd.DataFrame(rows)
display(normalisation_stats)


# Relative pigment-proxy contributions.
anth_n = normed["ARI2"]
cab_n = normed["CIRE"]
ccar_n = normed["CRI2"]

pigment_input_valid = (
    np.isfinite(ds["ARI2"].values)
    & np.isfinite(ds["CIRE"].values)
    & np.isfinite(ds["CRI2"].values)
)

pigment_total = anth_n + cab_n + ccar_n
relative_valid = (
    pigment_input_valid
    & np.isfinite(pigment_total)
    & (pigment_total > EPS)
)

# anth_rel = np.where(relative_valid, anth_n / pigment_total, np.nan).astype("float32")
# cab_rel = np.where(relative_valid, cab_n / pigment_total, np.nan).astype("float32")
# ccar_rel = np.where(relative_valid, ccar_n / pigment_total, np.nan).astype("float32")

# ✅ AFTER — np.divide with `where=` only touches valid pixels
anth_rel = np.full(anth_n.shape, np.nan, dtype="float32")
cab_rel  = np.full(cab_n.shape,  np.nan, dtype="float32")
ccar_rel = np.full(ccar_n.shape, np.nan, dtype="float32")

np.divide(anth_n, pigment_total, out=anth_rel, where=relative_valid)
np.divide(cab_n,  pigment_total, out=cab_rel,  where=relative_valid)
np.divide(ccar_n, pigment_total, out=ccar_rel, where=relative_valid)



ds["anth_rel"] = (("y", "x"), anth_rel)
ds["cab_rel"] = (("y", "x"), cab_rel)
ds["ccar_rel"] = (("y", "x"), ccar_rel)

ds["anth_rel"].attrs.update({
    "long_name": "Relative anthocyanin proxy contribution",
    "source_index": "ARI2",
})
ds["cab_rel"].attrs.update({
    "long_name": "Relative chlorophyll proxy contribution",
    "source_index": "CIRE",
})
ds["ccar_rel"].attrs.update({
    "long_name": "Relative carotenoid proxy contribution",
    "source_index": "CRI2",
})

relative_sum = anth_rel + cab_rel + ccar_rel
valid_sum = relative_sum[np.isfinite(relative_sum)]
print(
    "Relative-component sum: "
    f"min={valid_sum.min():.6f}, "
    f"mean={valid_sum.mean():.6f}, "
    f"max={valid_sum.max():.6f}"
)


# True-colour reference: R=668 nm, G=560 nm, B=475 nm.
rgb_ref = np.stack([
    pstretch(ds["red668"].values.astype("float32")),
    pstretch(ds["green560"].values.astype("float32")),
    pstretch(ds["blue"].values.astype("float32")),
], axis=2)
rgb_ref_uint8 = (rgb_ref * 255).astype(np.uint8)


## 4. Create the pigment composite

- Red = ARI2 relative contribution (anthocyanin proxy)
- Green = CIRE relative contribution (chlorophyll proxy)
- Blue = CRI2 relative contribution (carotenoid proxy)


In [ ]:
# Relative pigment-abundance channels: R=ARI2, G=CIRE, B=CRI2
ari = ds["anth_rel"].values
cire = ds["cab_rel"].values
cri = ds["ccar_rel"].values
valid = np.isfinite(ari) & np.isfinite(cire) & np.isfinite(cri)

pigment_rgb = np.stack([ari, cire, cri], axis=2)
pigment_rgb = np.clip(np.nan_to_num(pigment_rgb, nan=0.0), 0, 1)
pigment_uint8 = (pigment_rgb * 255).astype(np.uint8)
alpha = np.where(valid, 255, 0).astype(np.uint8)
pigment_rgba = np.dstack([pigment_uint8, alpha])

projected_context = projected_plot_context(ds)
relative_values = np.stack([ari, cire, cri], axis=-1)
unit = projected_context["unit_label"]

fig = go.Figure(
    make_projected_image(
        ds,
        pigment_rgba,
        context=projected_context,
        customdata=relative_values,
        hovertemplate=(
            f"Easting: %{{x:,.3f}} {unit}<br>"
            f"Northing: %{{y:,.3f}} {unit}<br>"
            "ARI2 relative abundance: %{customdata[0]:.3f}<br>"
            "CIRE relative abundance: %{customdata[1]:.3f}<br>"
            "CRI2 relative abundance: %{customdata[2]:.3f}"
            "<extra></extra>"
        ),
    )
)
apply_projected_axes(
    fig,
    projected_context,
    row=1,
    col=1,
    axis_number=1,
)
fig.update_layout(
    title=dict(
        text=(
            "Relative Pigment-Abundance Map"
            "<br><sup>Red = ARI2 · Green = CIRE · Blue = CRI2</sup>"
        ),
        x=0.5,
        xanchor="center",
    ),
    template="plotly_white",
    width=750,
    height=650,
    margin=dict(l=90, r=40, t=100, b=80),
)
fig.show(config={"scrollZoom": True, "displaylogo": False})


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def minmax_normalise(array):
    """Min-max normalise valid pixels to 0-1."""
    array = np.asarray(array, dtype=np.float32)
    output = np.full(array.shape, np.nan, dtype=np.float32)

    valid = np.isfinite(array)
    if not valid.any():
        return output

    minimum = np.nanmin(array)
    maximum = np.nanmax(array)

    if maximum > minimum:
        output[valid] = (
            (array[valid] - minimum) /
            (maximum - minimum)
        )
    else:
        output[valid] = 0

    return output


# True-colour composite: R=668 nm, G=560 nm, B=Blue
rgb_true = np.dstack([
    minmax_normalise(ds["red668"].values),
    minmax_normalise(ds["green560"].values),
    minmax_normalise(ds["blue"].values),
])

rgb_valid = np.all(np.isfinite(rgb_true), axis=2)
rgb_true = np.nan_to_num(rgb_true, nan=0.0)
rgb_true[~rgb_valid] = 0


# Relative pigment channels: R=ARI2, G=CIRE, B=CRI2
anth_rel = ds["anth_rel"].values
chl_rel = ds["cab_rel"].values
carotenoid_rel = ds["ccar_rel"].values

rgb_pigment = np.dstack([
    anth_rel,
    chl_rel,
    carotenoid_rel,
])

pigment_valid = np.all(np.isfinite(rgb_pigment), axis=2)
rgb_pigment = np.nan_to_num(rgb_pigment, nan=0.0)
rgb_pigment = np.clip(rgb_pigment, 0, 1)
rgb_pigment[~pigment_valid] = 0


fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 6),
    constrained_layout=True,
)

axes[0].imshow(rgb_true, origin="upper", interpolation="nearest", aspect="equal")
axes[0].set_title("True RGB")
axes[0].axis("off")

axes[1].imshow(rgb_pigment, origin="upper", interpolation="nearest", aspect="equal")
axes[1].set_title(
    "Relative Pigment Composite\n"
    "Anthocyanin (ARI2)=R, Chlorophyll (CIRE)=G, Carotenoid (CRI2)=B"
)
axes[1].axis("off")

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def minmax_normalise(array):
    """Normalise finite values to 0-1 while retaining NaN pixels."""
    array = np.asarray(array, dtype=np.float32)
    output = np.full(array.shape, np.nan, dtype=np.float32)

    valid = np.isfinite(array)
    if not valid.any():
        return output

    minimum = np.nanmin(array)
    maximum = np.nanmax(array)

    if maximum > minimum:
        output[valid] = (
            (array[valid] - minimum) /
            (maximum - minimum)
        )
    else:
        output[valid] = 0.0

    return output


# True-colour RGB
rgb_true = np.dstack([
    minmax_normalise(ds["red668"].values),
    minmax_normalise(ds["green560"].values),
    minmax_normalise(ds["blue"].values),
])

rgb_true = np.nan_to_num(
    rgb_true,
    nan=1.0,
    posinf=1.0,
    neginf=1.0,
)
rgb_true = np.clip(rgb_true, 0, 1)


# Vegetation mask from the original NDVI values
# Vegetation mask and threshold come from Section 2.5
# NDVI_THRESHOLD and ndvi_mask are already computed above.
veg_mask = ndvi_mask  # alias for clarity


# Relative pigment RGB: R=ARI2, G=CIRE, B=CRI2
anthocyanin = ds["anth_rel"].values
chlorophyll = ds["cab_rel"].values
carotenoid = ds["ccar_rel"].values

pigment_valid = (
    np.isfinite(anthocyanin)
    & np.isfinite(chlorophyll)
    & np.isfinite(carotenoid)
)

rgb_pigment = np.dstack([
    anthocyanin,
    chlorophyll,
    carotenoid,
])
rgb_pigment = np.clip(rgb_pigment, 0, 1)

display_mask = veg_mask & pigment_valid

# White background for non-vegetation and invalid pixels
rgb_pigment_masked = np.ones_like(rgb_pigment)
rgb_pigment_masked[display_mask] = rgb_pigment[display_mask]


fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 6),
    constrained_layout=True,
)

axes[0].imshow(rgb_true, origin="upper", interpolation="nearest", aspect="equal")
axes[0].set_title("True RGB")
axes[0].axis("off")

axes[1].imshow(
    rgb_pigment_masked,
    origin="upper",
    interpolation="nearest",
    aspect="equal",
)
axes[1].set_title(
    f"Relative Pigment Composite — NDVI > {NDVI_THRESHOLD:.2f}\n"
    "Anthocyanin (ARI2)=R, Chlorophyll (CIRE)=G, Carotenoid (CRI2)=B"
)
axes[1].axis("off")

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def minmax_normalise(array, mask=None):
    """Normalise selected finite pixels to the range 0-1."""
    array = np.asarray(array, dtype=np.float32)

    valid = np.isfinite(array)
    if mask is not None:
        valid &= mask

    output = np.full(array.shape, np.nan, dtype=np.float32)

    if not valid.any():
        return output

    minimum = float(np.min(array[valid]))
    maximum = float(np.max(array[valid]))

    if maximum > minimum:
        output[valid] = (
            (array[valid] - minimum) /
            (maximum - minimum)
        )
    else:
        output[valid] = 0.0

    return output


# True-colour RGB
rgb_true = np.dstack([
    minmax_normalise(ds["red668"].values),
    minmax_normalise(ds["green560"].values),
    minmax_normalise(ds["blue"].values),
])

rgb_true_display = np.nan_to_num(
    rgb_true,
    nan=1.0,
    posinf=1.0,
    neginf=1.0,
)
rgb_true_display = np.clip(rgb_true_display, 0, 1)


# Vegetation mask using original NDVI values
# Vegetation mask and threshold come from Section 2.5
# NDVI_THRESHOLD and ndvi_mask are already computed above.
veg_mask = ndvi_mask  # alias for clarity


# Relative pigment RGB: R=ARI2, G=CIRE, B=CRI2
anthocyanin = ds["anth_rel"].values
chlorophyll = ds["cab_rel"].values
carotenoid = ds["ccar_rel"].values

rgb_pigment = np.dstack([
    anthocyanin,
    chlorophyll,
    carotenoid,
])
rgb_pigment = np.clip(rgb_pigment, 0, 1)

pigment_valid = np.all(np.isfinite(rgb_pigment), axis=2)
display_mask = veg_mask & pigment_valid

# Analysis/export version: non-vegetation pixels remain NaN
rgb_pigment_masked = np.full_like(
    rgb_pigment,
    np.nan,
    dtype=np.float32,
)
rgb_pigment_masked[display_mask] = rgb_pigment[display_mask]

# Rasterio band order: bands, rows, columns
rgb_out = np.moveaxis(
    rgb_pigment_masked,
    -1,
    0,
).astype(np.float32)
valid_output = display_mask.copy()

# Display version
rgb_pigment_display = np.nan_to_num(rgb_pigment_masked, nan=0.0)

alpha = 0.50
alpha_mask = np.zeros(veg_mask.shape, dtype=np.float32)
alpha_mask[display_mask] = alpha


fig, ax = plt.subplots(figsize=(8, 8))

ax.imshow(rgb_true_display, origin="upper", interpolation="nearest", aspect="equal")
ax.imshow(
    rgb_pigment_display,
    origin="upper",
    interpolation="nearest",
    aspect="equal",
    alpha=alpha_mask,
)

ax.set_title(
    "Relative Pigment Composite over True RGB\n"
    f"NDVI > {NDVI_THRESHOLD:.2f}; "
    "Anthocyanin (ARI2)=R, Chlorophyll (CIRE)=G, "
    "Carotenoid (CRI2)=B"
)

ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# Full-resolution arrays
pigment_rgb = np.stack([
    ds["anth_rel"].values,
    ds["cab_rel"].values,
    ds["ccar_rel"].values,
], axis=2)
pigment_valid = np.all(np.isfinite(pigment_rgb), axis=2)
pigment_rgb = np.clip(np.nan_to_num(pigment_rgb, nan=0.0), 0, 1)
pigment_rgb[~pigment_valid] = 0
pigment_display = (pigment_rgb * 255).astype(np.uint8)

ndvi_display = np.where(
    np.isfinite(ds["NDVI"].values),
    normed["NDVI"],
    np.nan,
)
rgb_display = rgb_ref_uint8
projected_context = projected_plot_context(ds)

assert rgb_display.shape[:2] == ndvi_display.shape
assert pigment_display.shape[:2] == ndvi_display.shape

fig_compare = make_subplots(
    rows=1,
    cols=3,
    column_widths=[1, 1, 1],
    horizontal_spacing=0.04,
    subplot_titles=[
        "RGB true-colour",
        "NDVI (normalised)",
        "Relative pigment abundance",
    ],
)
fig_compare.add_trace(
    make_projected_image(ds, rgb_display, context=projected_context),
    row=1,
    col=1,
)
fig_compare.add_trace(
    make_projected_heatmap(
        ds,
        array=ndvi_display,
        context=projected_context,
        colorscale="YlGn",
        zmin=0,
        zmax=1,
        showscale=False,
        hover_name="NDVI",
        value_label="NDVI (normalised)",
    ),
    row=1,
    col=2,
)
fig_compare.add_trace(
    make_projected_image(ds, pigment_display, context=projected_context),
    row=1,
    col=3,
)

for column in range(1, 4):
    apply_projected_axes(
        fig_compare,
        projected_context,
        row=1,
        col=column,
        axis_number=column,
        x_title=True,
        y_title=column == 1,
    )
fig_compare.update_xaxes(matches="x", row=1, col=2)
fig_compare.update_xaxes(matches="x", row=1, col=3)
fig_compare.update_yaxes(matches="y", row=1, col=2)
fig_compare.update_yaxes(matches="y", row=1, col=3)

fig_compare.update_layout(
    title=dict(
        text=(
            "RGB Reference, NDVI and Relative Pigment Abundance"
            "<br><sup>Pigment composite: "
            "Red = ARI2 · Green = CIRE · Blue = CRI2</sup>"
        ),
        x=0.5,
        xanchor="center",
    ),
    template="plotly_white",
    width=1300,
    height=600,
    showlegend=False,
    dragmode="zoom",
    hovermode="closest",
    margin=dict(l=90, r=30, t=105, b=85),
)
fig_compare.show(
    config={
        "scrollZoom": True,
        "displaylogo": False,
        "modeBarButtonsToAdd": ["drawrect", "eraseshape"],
    }
)


## 5. Masked-data threshold explorer

The NDVI mask has already been applied. The explorer below shows
how a **secondary threshold** on the normalised index further
subsets the already-masked vegetation pixels.


In [ ]:
# NOTE: ds indices are already NDVI-masked (non-vegetation = NaN).
# The slider below sub-selects within the vegetation pixels only.

th_idx = widgets.Dropdown(
    options=ANALYSIS_INDICES,
    value="NDVI",
    description="Index:",
)
th_val = widgets.FloatSlider(
    value=0.20,
    min=0,
    max=1,
    step=0.01,
    description="Threshold:",
    continuous_update=True,
    readout_format=".2f",
)
threshold_output = widgets.Output()
threshold_context = projected_plot_context(ds)


def render_threshold(change=None):
    with threshold_output:
        clear_output(wait=True)
        name = th_idx.value
        info = INDEX_INFO[name]
        arr = normed[name]
        original = ds[name].values

        valid_mask = np.isfinite(original) & np.isfinite(arr)
        selected_mask = valid_mask & (arr >= th_val.value)
        masked = np.where(selected_mask, arr, np.nan)
        kept = int(selected_mask.sum())
        total = int(valid_mask.sum())
        pct = 100 * kept / total if total else 0.0

        fig = go.Figure(
            make_projected_heatmap(
                ds,
                array=masked,
                context=threshold_context,
                colorscale=info["cmap"],
                zmin=0,
                zmax=1,
                showscale=True,
                colorbar=dict(title=f"{name} (norm.)", thickness=14),
                hover_name=name,
                value_label=f"{name} (normalised)",
            )
        )
        apply_projected_axes(
            fig,
            threshold_context,
            row=1,
            col=1,
            axis_number=1,
        )
        fig.update_layout(
            title=(
                f"{name} ≥ {th_val.value:.2f} — "
                f"{kept:,} of {total:,} valid pixels ({pct:.1f}%)"
            ),
            template="plotly_white",
            width=850,
            height=650,
            showlegend=False,
            margin=dict(l=90, r=100, t=80, b=80),
        )
        fig.show(config={"scrollZoom": True, "displaylogo": False})

        display(Markdown(
            f"**{name}:** {info['full_name']}  \n"
            f"**Interpretation:** {info['interpretation']}"
        ))


for control in (th_idx, th_val):
    control.observe(render_threshold, names="value")
display(widgets.HBox([th_idx, th_val]), threshold_output)
render_threshold()


In [ ]:
# NOTE: All arrays are already NDVI-masked. The slider here controls
# an additional sub-threshold within vegetation pixels only.

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown


# ============================================================
# Convert xarray DataArray or NumPy array consistently
# ============================================================

def as_array(data):
    if hasattr(data, "values"):
        data = data.values
    return np.asarray(data, dtype=np.float32)


# ============================================================
# Controls
# ============================================================

# NDVI remains fixed on the left.
# The dropdown controls only the right panel.
comparison_indices = [
    name for name in ANALYSIS_INDICES
    if name != "NDVI"
]

if not comparison_indices:
    raise ValueError(
        "ANALYSIS_INDICES must contain at least one index besides NDVI."
    )

comparison_index = widgets.Dropdown(
    options=comparison_indices,
    value=(
        "CIRE"
        if "CIRE" in comparison_indices
        else comparison_indices[0]
    ),
    description="Right index:",
)

ndvi_threshold = widgets.FloatSlider(
    value=0.20,
    min=0,
    max=1,
    step=0.01,
    description="NDVI threshold:",
    continuous_update=True,
    readout_format=".2f",
)

threshold_output = widgets.Output()

# Shared projected coordinates
threshold_context = projected_plot_context(ds)


# ============================================================
# Interactive plotting function
# ============================================================

def render_threshold_comparison(change=None):
    with threshold_output:
        clear_output(wait=True)

        right_name = comparison_index.value
        threshold = ndvi_threshold.value

        # ----------------------------------------------------
        # Read normalised arrays
        # ----------------------------------------------------

        # Fixed left panel
        ndvi = as_array(normed["NDVI"])

        # Dropdown-controlled right panel
        right_index = as_array(normed[right_name])

        # Original arrays used for validity checking
        ndvi_original = as_array(ds["NDVI"])
        right_original = as_array(ds[right_name])

        # Confirm matching dimensions
        if ndvi.shape != right_index.shape:
            raise ValueError(
                f"NDVI shape {ndvi.shape} does not match "
                f"{right_name} shape {right_index.shape}."
            )

        # ----------------------------------------------------
        # NDVI threshold selection
        # ----------------------------------------------------

        ndvi_valid = (
            np.isfinite(ndvi_original)
            & np.isfinite(ndvi)
        )

        right_valid = (
            np.isfinite(right_original)
            & np.isfinite(right_index)
        )

        # NDVI controls the selection for both panels
        selected = (
            ndvi_valid
            & (ndvi >= threshold)
        )

        # Left: thresholded NDVI
        ndvi_display = np.where(
            selected,
            ndvi,
            np.nan,
        )

        # Right: corresponding pixels from selected index
        right_display = np.where(
            selected & right_valid,
            right_index,
            np.nan,
        )

        kept = int(selected.sum())
        total = int(ndvi_valid.sum())
        percent = 100 * kept / total if total else 0.0

        # ----------------------------------------------------
        # Create two panels
        # ----------------------------------------------------

        fig = make_subplots(
            rows=1,
            cols=2,
            column_widths=[0.5, 0.5],
            horizontal_spacing=0.12,
            subplot_titles=[
                f"NDVI ≥ {threshold:.2f}",
                f"{right_name} at NDVI-selected pixels",
            ],
        )

        # ----------------------------------------------------
        # Left panel: NDVI
        # ----------------------------------------------------

        fig.add_trace(
            make_projected_heatmap(
                ds,
                array=ndvi_display,
                context=threshold_context,
                colorscale=INDEX_INFO["NDVI"].get(
                    "cmap",
                    "RdYlGn",
                ),
                zmin=0,
                zmax=1,
                showscale=True,
                colorbar=dict(
                    title="NDVI",
                    x=0.455,
                    xanchor="left",
                    y=0.5,
                    len=0.82,
                    thickness=12,
                ),
                hover_name="NDVI",
                value_label="NDVI (normalised)",
            ),
            row=1,
            col=1,
        )

        # ----------------------------------------------------
        # Right panel: dropdown-selected index
        # ----------------------------------------------------

        fig.add_trace(
            make_projected_heatmap(
                ds,
                array=right_display,
                context=threshold_context,
                colorscale=INDEX_INFO[right_name].get(
                    "cmap",
                    "Viridis",
                ),
                zmin=0,
                zmax=1,
                showscale=True,
                colorbar=dict(
                    title=right_name,
                    x=1.01,
                    xanchor="left",
                    y=0.5,
                    len=0.82,
                    thickness=12,
                ),
                hover_name=right_name,
                value_label=f"{right_name} (normalised)",
            ),
            row=1,
            col=2,
        )

        # ----------------------------------------------------
        # Projected coordinate axes
        # ----------------------------------------------------

        apply_projected_axes(
            fig,
            threshold_context,
            row=1,
            col=1,
            axis_number=1,
        )

        apply_projected_axes(
            fig,
            threshold_context,
            row=1,
            col=2,
            axis_number=2,
        )

        # Synchronise zooming and panning
        fig.update_xaxes(
            matches="x",
            row=1,
            col=2,
        )

        fig.update_yaxes(
            matches="y",
            row=1,
            col=2,
        )

        # ----------------------------------------------------
        # Layout
        # ----------------------------------------------------

        fig.update_layout(
            title=dict(
                text=(
                    f"NDVI ≥ {threshold:.2f}: "
                    f"{kept:,} of {total:,} valid pixels "
                    f"({percent:.1f}%)"
                ),
                x=0.5,
                xanchor="center",
            ),
            template="plotly_white",
            width=1200,
            height=600,
            showlegend=False,
            margin=dict(
                l=90,
                r=110,
                t=100,
                b=80,
            ),
            hoverlabel=dict(
                bgcolor="white",
                font=dict(
                    color="black",
                    size=12,
                ),
            ),
        )

        fig.show(
            config={
                "scrollZoom": True,
                "displaylogo": False,
            }
        )

        # ----------------------------------------------------
        # Index description
        # ----------------------------------------------------

        info = INDEX_INFO[right_name]

        display(
            Markdown(
                f"**Right panel: {right_name} — "
                f"{info['full_name']}**  \n"
                f"**Interpretation:** {info['interpretation']}"
            )
        )


# ============================================================
# Connect controls
# ============================================================

for control in (
    comparison_index,
    ndvi_threshold,
):
    control.observe(
        render_threshold_comparison,
        names="value",
    )


# ============================================================
# Display
# ============================================================

display(
    widgets.HBox([
        comparison_index,
        ndvi_threshold,
    ]),
    threshold_output,
)

render_threshold_comparison()

## 6. Pigment abundance composite on a basemap


In [ ]:
import folium
import rasterio
from rasterio.warp import transform_bounds

with rasterio.open(RASTER_PATH) as src:
    west, south, east, north = transform_bounds(
        src.crs,
        "EPSG:4326",
        *src.bounds,
    )

map_step = max(1, max(pigment_uint8.shape[:2]) // 500)
pigment_uri = array_to_png_b64(pigment_uint8[::map_step, ::map_step])

pigment_map = folium.Map(
    location=[(south + north) / 2, (west + east) / 2],
    zoom_start=17,
)
folium.TileLayer("openstreetmap", name="OpenStreetMap").add_to(pigment_map)
folium.TileLayer(
    tiles=(
        "https://server.arcgisonline.com/ArcGIS/rest/services/"
        "World_Imagery/MapServer/tile/{z}/{y}/{x}"
    ),
    attr="Esri World Imagery",
    name="Esri Satellite",
).add_to(pigment_map)
folium.raster_layers.ImageOverlay(
    image=pigment_uri,
    bounds=[[south, west], [north, east]],
    opacity=0.88,
    name="Pigment composite (R=ARI2, G=CIRE, B=CRI2)",
    interactive=True,
).add_to(pigment_map)
folium.LayerControl(collapsed=False).add_to(pigment_map)
pigment_map.fit_bounds([[south, west], [north, east]])
pigment_map


In [ ]:
import base64
import struct
import zlib

import numpy as np
import folium
import rasterio
from rasterio.warp import transform_bounds


def rgba_to_png_b64(rgba_uint8):
    """Convert an (H, W, 4) uint8 RGBA array to a PNG data URI."""
    height, width, channels = rgba_uint8.shape
    if channels != 4 or rgba_uint8.dtype != np.uint8:
        raise ValueError("Expected an (H, W, 4) uint8 RGBA array.")

    def chunk(chunk_type, data):
        payload = chunk_type + data
        return (
            struct.pack(">I", len(data))
            + payload
            + struct.pack(">I", zlib.crc32(payload) & 0xFFFFFFFF)
        )

    signature = b"\x89PNG\r\n\x1a\n"
    # PNG colour type 6 = RGBA.
    ihdr = chunk(
        b"IHDR",
        struct.pack(">IIBBBBB", width, height, 8, 6, 0, 0, 0),
    )
    raw = b"".join(b"\x00" + rgba_uint8[row].tobytes() for row in range(height))
    idat = chunk(b"IDAT", zlib.compress(raw, level=6))
    iend = chunk(b"IEND", b"")
    png = signature + ihdr + idat + iend
    return "data:image/png;base64," + base64.b64encode(png).decode("ascii")


valid_pigment = np.all(np.isfinite(rgb_pigment_masked), axis=2)

pigment_rgb_clean = np.nan_to_num(
    rgb_pigment_masked,
    nan=0.0,
    posinf=1.0,
    neginf=0.0,
)
pigment_rgb_clean = np.clip(pigment_rgb_clean, 0, 1)
pigment_uint8 = np.round(pigment_rgb_clean * 255).astype(np.uint8)

alpha_channel = np.where(valid_pigment, 255, 0).astype(np.uint8)
pigment_rgba = np.dstack([pigment_uint8, alpha_channel])


with rasterio.open(RASTER_PATH) as src:
    if src.crs is None:
        raise ValueError("RASTER_PATH does not contain CRS information.")

    if src.height != pigment_rgba.shape[0] or src.width != pigment_rgba.shape[1]:
        raise ValueError(
            "Pigment image dimensions do not match DATA_PATH.\n"
            f"Raster: {src.height} × {src.width}\n"
            f"Image: {pigment_rgba.shape[0]} × {pigment_rgba.shape[1]}"
        )

    west, south, east, north = transform_bounds(
        src.crs,
        "EPSG:4326",
        *src.bounds,
        densify_pts=21,
    )


map_step = max(1, max(pigment_rgba.shape[:2]) // 500)
pigment_rgba_map = pigment_rgba[::map_step, ::map_step]
pigment_uri = rgba_to_png_b64(pigment_rgba_map)


pigment_map = folium.Map(
    location=[(south + north) / 2, (west + east) / 2],
    tiles=None,
    zoom_start=17,
    control_scale=True,
)

folium.TileLayer("openstreetmap", name="OpenStreetMap").add_to(pigment_map)
folium.TileLayer(
    tiles=(
        "https://server.arcgisonline.com/ArcGIS/rest/services/"
        "World_Imagery/MapServer/tile/{z}/{y}/{x}"
    ),
    attr="Esri World Imagery",
    name="Esri Satellite",
).add_to(pigment_map)

folium.raster_layers.ImageOverlay(
    image=pigment_uri,
    bounds=[[south, west], [north, east]],
    opacity=0.88,
    name=(
        "Relative pigments "
        "(R=ARI2 anthocyanin, G=CIRE chlorophyll, B=CRI2 carotenoid)"
    ),
    interactive=True,
).add_to(pigment_map)

folium.LayerControl(collapsed=False).add_to(pigment_map)
pigment_map.fit_bounds([[south, west], [north, east]])
pigment_map


In [ ]:
# Run once if SciPy is unavailable:
# %pip install scipy matplotlib pandas

# The diagonal panels show smoothed index distributions. 
# Each lower panel shows the smoothed joint pixel density and its Pearson correlation coefficient. 
# With PERCENTILE_RANGE = (2, 98), all pixels within the joint P2–P98 limits are used. 
# Set it to None to retain every complete pixel.

    

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.colors import LinearSegmentedColormap, PowerNorm
from scipy.ndimage import gaussian_filter, gaussian_filter1d
from IPython.display import display


# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

INDEX_NAMES = [
    "NDVI",
    "CIRE",
    "ARI2",
    "CRI2",
]

# Use (2, 98), (5, 95), or None for no percentile filtering
PERCENTILE_RANGE = (2, 98)

HISTOGRAM_BINS = 100
DENSITY_BINS = 150
SMOOTHING = 1.3


# ------------------------------------------------------------
# Collect pixels valid in all four indices
# ------------------------------------------------------------

arrays = [
    ds[name].values.ravel().astype(np.float32)
    for name in INDEX_NAMES
]

stack = np.column_stack(arrays)

# Complete-case mask
complete = np.all(np.isfinite(stack), axis=1)
data = stack[complete]

if data.size == 0:
    raise ValueError(
        "There are no pixels valid in all four indices."
    )


# ------------------------------------------------------------
# Optional percentile filtering
# ------------------------------------------------------------

if PERCENTILE_RANGE is not None:
    percentile_low, percentile_high = PERCENTILE_RANGE

    lower = np.percentile(
        data,
        percentile_low,
        axis=0,
    )
    upper = np.percentile(
        data,
        percentile_high,
        axis=0,
    )

    within_percentiles = np.all(
        (data >= lower)
        & (data <= upper),
        axis=1,
    )

    data = data[within_percentiles]

    filter_label = (
        f"P{percentile_low}–P{percentile_high}"
    )
else:
    filter_label = "all valid values"

if data.shape[0] < 2:
    raise ValueError(
        "Too few pixels remain after percentile filtering."
    )

print(
    f"Pixels used: {data.shape[0]:,} "
    f"({filter_label})"
)


# ------------------------------------------------------------
# Pearson correlation matrix using all retained pixels
# ------------------------------------------------------------

correlation = np.corrcoef(
    data,
    rowvar=False,
)

correlation_table = pd.DataFrame(
    correlation,
    index=INDEX_NAMES,
    columns=INDEX_NAMES,
).round(3)

display(correlation_table)


# ------------------------------------------------------------
# Plot ranges
# ------------------------------------------------------------

plot_ranges = []

for column in range(len(INDEX_NAMES)):
    minimum = float(np.min(data[:, column]))
    maximum = float(np.max(data[:, column]))

    padding = 0.02 * (maximum - minimum)

    if padding == 0:
        padding = 0.01

    plot_ranges.append(
        (
            minimum - padding,
            maximum + padding,
        )
    )


# ------------------------------------------------------------
# Sequential density colour scale
# ------------------------------------------------------------

density_cmap = LinearSegmentedColormap.from_list(
    "density_orange",
    [
        "#ffffff",
        "#fee8dc",
        "#fdbb84",
        "#ef6548",
        "#990000",
    ],
)


# ------------------------------------------------------------
# Lower-triangular pair plot
# ------------------------------------------------------------

number_indices = len(INDEX_NAMES)

fig, axes = plt.subplots(
    number_indices,
    number_indices,
    figsize=(10, 10),
    constrained_layout=True,
)

for row in range(number_indices):
    for column in range(number_indices):

        ax = axes[row, column]

        # Upper triangle remains empty
        if column > row:
            ax.axis("off")
            continue

        x = data[:, column]

        # ----------------------------------------------------
        # Diagonal: smoothed one-dimensional density
        # ----------------------------------------------------

        if row == column:
            density, edges = np.histogram(
                x,
                bins=HISTOGRAM_BINS,
                range=plot_ranges[column],
                density=True,
            )

            centres = (
                edges[:-1] + edges[1:]
            ) / 2

            density_smooth = gaussian_filter1d(
                density.astype(np.float64),
                sigma=SMOOTHING,
            )

            ax.fill_between(
                centres,
                density_smooth,
                color="#d8b9a5",
                alpha=0.80,
            )

            ax.plot(
                centres,
                density_smooth,
                color="#9f765f",
                linewidth=1.0,
            )

            ax.set_xlim(plot_ranges[column])
            ax.set_yticks([])

        # ----------------------------------------------------
        # Lower triangle: smoothed two-dimensional density
        # ----------------------------------------------------

        else:
            y = data[:, row]

            density_2d, x_edges, y_edges = np.histogram2d(
                x,
                y,
                bins=DENSITY_BINS,
                range=[
                    plot_ranges[column],
                    plot_ranges[row],
                ],
            )

            # Smooth the binned density to produce a KDE-like map
            density_2d = gaussian_filter(
                density_2d.astype(np.float64),
                sigma=SMOOTHING,
            )

            # Hide cells without observations
            density_masked = np.ma.masked_less_equal(
                density_2d.T,
                0,
            )

            maximum_density = float(
                np.max(density_2d)
            )

            ax.pcolormesh(
                x_edges,
                y_edges,
                density_masked,
                cmap=density_cmap,
                norm=PowerNorm(
                    gamma=0.45,
                    vmin=0,
                    vmax=maximum_density,
                ),
                shading="auto",
                rasterized=True,
            )

            ax.set_xlim(plot_ranges[column])
            ax.set_ylim(plot_ranges[row])

            # Pearson correlation annotation
            ax.text(
                0.04,
                0.94,
                f"r = {correlation[row, column]:.2f}",
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontsize=11,
                color="#b2182b",
                bbox=dict(
                    facecolor="white",
                    edgecolor="none",
                    alpha=0.75,
                    pad=1.5,
                ),
            )

        # ----------------------------------------------------
        # Labels and styling
        # ----------------------------------------------------

        if row == number_indices - 1:
            ax.set_xlabel(
                INDEX_NAMES[column],
                fontsize=11,
            )
        else:
            ax.tick_params(
                axis="x",
                labelbottom=False,
            )

        if column == 0 and row > 0:
            ax.set_ylabel(
                INDEX_NAMES[row],
                fontsize=11,
            )
        elif column > 0:
            ax.tick_params(
                axis="y",
                labelleft=False,
            )

        ax.tick_params(
            axis="both",
            labelsize=9,
        )

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_color("#aaaaaa")
        ax.spines["bottom"].set_color("#aaaaaa")


fig.suptitle(
    (
        "Spectral-Index Pairwise Density and Pearson Correlation\n"
        f"{filter_label}; n={data.shape[0]:,} pixels"
    ),
    fontsize=14,
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from scipy.ndimage import gaussian_filter1d

INDEX_NAMES = ["NDVI", "CIRE", "ARI2", "CRI2"]

BLUE = "#2166ac"
LIGHT_BLUE = "#c6dbef"

# White/pale blue for isolated points; dark blue only for dense regions
blue_density = LinearSegmentedColormap.from_list(
    "blue_density",
    [
        "#f7fbff",
        "#deebf7",
        "#c6dbef",
        "#9ecae1",
        "#6baed6",
        "#4292c6",
        "#2166ac",
    ],
)

# Previous orange colours — disabled
# ORANGE = "#d95f0e"
# orange_density = LinearSegmentedColormap.from_list(
#     "orange_density",
#     ["#fff5eb", "#fee6ce", "#fdae6b", "#e6550d", "#a63603"],
# )

# Flatten the original index arrays
index_data = {
    name: ds[name].values.ravel().astype(np.float64)
    for name in INDEX_NAMES
}


def point_density(x, y, bins=180):
    """Assign every point the count of its local 2-D histogram cell."""

    counts, x_edges, y_edges = np.histogram2d(
        x,
        y,
        bins=bins,
    )

    x_bin = np.searchsorted(x_edges, x, side="right") - 1
    y_bin = np.searchsorted(y_edges, y, side="right") - 1

    x_bin = np.clip(x_bin, 0, counts.shape[0] - 1)
    y_bin = np.clip(y_bin, 0, counts.shape[1] - 1)

    return counts[x_bin, y_bin]


n_indices = len(INDEX_NAMES)

fig, axes = plt.subplots(
    n_indices,
    n_indices,
    figsize=(9, 9),
)

fig.patch.set_facecolor("white")


for row in range(n_indices):
    for col in range(n_indices):

        ax = axes[row, col]
        ax.set_facecolor("white")

        # Remove upper triangle
        if col > row:
            ax.axis("off")
            continue

        x_name = INDEX_NAMES[col]
        y_name = INDEX_NAMES[row]

        # ====================================================
        # Diagonal: marginal density
        # ====================================================
        if row == col:
            x = index_data[x_name]
            x = x[np.isfinite(x)]

            counts, edges = np.histogram(
                x,
                bins=120,
                density=True,
            )

            centres = (edges[:-1] + edges[1:]) / 2

            smooth_density = gaussian_filter1d(
                counts.astype(float),
                sigma=2,
            )

            ax.fill_between(
                centres,
                smooth_density,
                color=LIGHT_BLUE,
                alpha=0.60,
            )

            ax.plot(
                centres,
                smooth_density,
                color=BLUE,
                linewidth=0.9,
            )

            ax.set_xlim(np.min(x), np.max(x))
            ax.set_ylim(bottom=0)
            ax.set_yticks([])

        # ====================================================
        # Lower triangle: all points coloured by local density
        # ====================================================
        else:
            x_all = index_data[x_name]
            y_all = index_data[y_name]

            valid = np.isfinite(x_all) & np.isfinite(y_all)

            x = x_all[valid]
            y = y_all[valid]

            # Density calculated from every valid point
            density = point_density(x, y, bins=180)

            # Draw low-density points first and dense points last
            order = np.argsort(density)
            x = x[order]
            y = y[order]
            density = density[order]

            # Prevent a few exceptionally dense cells from
            # controlling the entire colour range
            density_max = max(
                2.0,
                float(np.percentile(density, 99.5)),
            )

            ax.scatter(
                x,
                y,
                c=density,
                cmap=blue_density,
                norm=LogNorm(
                    vmin=1,
                    vmax=density_max,
                    clip=True,
                ),
                s=0.30,
                alpha=1.0,
                linewidths=0,
                edgecolors="none",
                rasterized=True,
            )

            # Pearson correlation using every valid pair
            if (
                len(x) > 1
                and np.std(x) > 0
                and np.std(y) > 0
            ):
                correlation = np.corrcoef(x, y)[0, 1]
            else:
                correlation = np.nan

            ax.text(
                0.03,
                0.97,
                f"{correlation:.2f}",
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontsize=10,
                fontweight="bold",
                color=BLUE,
                bbox=dict(
                    facecolor="white",
                    edgecolor="none",
                    alpha=0.75,
                    pad=1,
                ),
            )

            ax.set_xlim(np.min(x), np.max(x))
            ax.set_ylim(np.min(y), np.max(y))

        # ====================================================
        # Axis labels and styling
        # ====================================================
        if row == n_indices - 1:
            ax.set_xlabel(x_name, fontsize=11)
        else:
            ax.tick_params(axis="x", labelbottom=False)

        if col == 0 and row > 0:
            ax.set_ylabel(y_name, fontsize=11)
        elif col > 0:
            ax.tick_params(axis="y", labelleft=False)

        ax.tick_params(
            axis="both",
            labelsize=8,
            length=3,
            colors="#333333",
        )

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_color("#999999")
        ax.spines["bottom"].set_color("#999999")
        ax.spines["left"].set_linewidth(0.7)
        ax.spines["bottom"].set_linewidth(0.7)


# Completely remove unused upper-triangle panels
for row in range(n_indices):
    for col in range(row + 1, n_indices):
        fig.delaxes(axes[row, col])


plt.subplots_adjust(
    left=0.10,
    right=0.98,
    bottom=0.09,
    top=0.98,
    wspace=0.08,
    hspace=0.08,
)

plt.show()

# Save pigment maps 

In [ ]:
from pathlib import Path

import numpy as np
import rasterio
from rasterio.enums import ColorInterp


project_dir = Path.cwd().resolve()

src_path = RASTER_PATH
out_path = OUTPUT_DIR/ "pigment_rgb_relative.tif"

out_path.parent.mkdir(parents=True, exist_ok=True)

if not src_path.exists():
    raise FileNotFoundError(f"Source raster not found: {src_path}")

print("Source:", src_path)
print("Output:", out_path)
out_path.parent.mkdir(parents=True, exist_ok=True)

bands, rows, cols = rgb_out.shape


# Read exact georeferencing from the source raster
with rasterio.open(src_path) as src:
    if src.height != rows or src.width != cols:
        raise ValueError(
            "The source raster dimensions do not match the output arrays.\n"
            f"Source: {src.height} rows × {src.width} columns\n"
            f"Output: {rows} rows × {cols} columns"
        )

    if src.crs is None:
        raise ValueError("The source raster has no CRS.")

    output_crs = src.crs
    output_transform = src.transform


# Export the relative pigment composite
nodata_value = -9999.0
valid_output = np.all(np.isfinite(rgb_out), axis=0)

rgb_out_export = rgb_out.astype(np.float32, copy=True)
rgb_out_export[:, ~valid_output] = nodata_value

profile = {
    "driver": "GTiff",
    "height": rows,
    "width": cols,
    "count": 3,
    "dtype": "float32",
    "crs": output_crs,
    "transform": output_transform,
    "nodata": nodata_value,
    "compress": "deflate",
    "predictor": 3,
    "tiled": True,
    "BIGTIFF": "IF_SAFER",
}

with rasterio.open(out_path, "w", **profile) as dst:
    dst.write(rgb_out_export)

    dst.colorinterp = (
        ColorInterp.red,
        ColorInterp.green,
        ColorInterp.blue,
    )

    dst.set_band_description(1, "Relative anthocyanin — ARI2 — Red")
    dst.set_band_description(2, "Relative chlorophyll — CIRE — Green")
    dst.set_band_description(3, "Relative carotenoid — CRI2 — Blue")

    dst.write_mask(valid_output.astype(np.uint8) * 255)

    dst.update_tags(
        product="Relative pigment RGB composite",
        red_channel="ARI2 relative anthocyanin",
        green_channel="CIRE relative chlorophyll",
        blue_channel="CRI2 relative carotenoid",
        normalisation="P2-P98 scaling followed by pixel-wise relative contribution",
        ndvi_threshold=str(NDVI_THRESHOLD),
    )


# Verify the saved GeoTIFF
with rasterio.open(out_path) as src:
    saved = src.read(masked=True)

    print(f"Saved: {out_path}")
    print("Shape:", saved.shape)
    print("CRS:", src.crs)
    print("Transform:", src.transform)
    print("Bounds:", src.bounds)
    print("Descriptions:", src.descriptions)

    labels = [
        "Anthocyanin/ARI2",
        "Chlorophyll/CIRE",
        "Carotenoid/CRI2",
    ]

    for index, label in enumerate(labels):
        values = saved[index].compressed()

        if values.size:
            print(
                f"{label}: "
                f"min={values.min():.4f}, "
                f"max={values.max():.4f}"
            )
        else:
            print(f"{label}: no valid pixels")


### Reading the pigment composite

The three channels are relative contributions and sum to approximately 1
for each valid pixel. Therefore, colour represents relative composition,
not total pigment concentration.

| Colour | Dominant relative signal | Interpretation |
|---|---|---|
| Red | High ARI2 contribution | Anthocyanin proxy dominates |
| Green | High CIRE contribution | Chlorophyll proxy dominates |
| Blue | High CRI2 contribution | Carotenoid proxy dominates |
| Yellow | ARI2 + CIRE | Anthocyanin and chlorophyll proxies are jointly strong |
| Cyan | CIRE + CRI2 | Chlorophyll and carotenoid proxies are jointly strong |
| Magenta | ARI2 + CRI2 | Anthocyanin and carotenoid proxies are jointly strong |
| White/transparent | Masked | Non-vegetation or invalid input |

## Activities

1. For CIRE, calculate the P2-P98 normalised value of a raw value by hand.
2. Select ARI2 in the threshold explorer. At what threshold does the
   selected area drop below 20%? Are the retained pixels clustered?
3. Compare blue, green, and red-dominant areas against the satellite layer.

The composite is a relative visual summary, not an estimate of absolute
biochemical concentration. Interpret patterns alongside field observations
and environmental context.
